In [1]:
# make and activate venv environment, manually ...

# !pip install torch==2.9.1+cu128 --index-url https://download.pytorch.org/whl/cu128
# !pip install pytorch-lightning~=2.0

# !pip install torch_geometric
# !pip install torch_cluster -f https://data.pyg.org/whl/torch-2.9.1+cu128.html  # no windows version for torch-2.9.1+cu130

# # loading pre-trained model requires numpy < 2
# !pip install -r requirements.txt

# !pip install pysdf>=0.1
# # getting an error like error C2039: 'high_resolution_clock': is not a member of 'std::chrono'?
# # fix these errors by adding "#include <chrono>" to the relevant files or installing this in a VS 2019 tools prompt or older

# # install executorch
# !pip install executorch

In [2]:
# capture test data for tracing 
# capture manually once while stepping through the code
# add breakpoints and run the code in debug mode, then execute the following lines in the watch window to save the test data for tracing
# relevant code lines are in poco_utils.py, functions generate_latent_representation and predict_from_latent

# encoding:
# torch.save((batch, network, network_latent_size, gen_subsample_manifold_iter, gen_subsample_manifold, None), 'tracing_test_data/generate_latent_representation_in.pt')
# torch.save(batch, 'tracing_test_data/get_data_poco_in.pt')
# torch.save(shape_data_poco, 'tracing_test_data/get_data_poco_out.pt')
# torch.save(data_partial, 'tracing_test_data/network_get_latent_in.pt')
# torch.save(partial_latent, 'tracing_test_data/network_get_latent_out.pt')
# torch.save((shape_data_poco, latent), 'tracing_test_data/generate_latent_representation_out.pt')

# decoding:
# torch.save((latent, network, pts_query, pts_raw_ms, num_pts_local, None), 'tracing_test_data/predict_from_latent_in.pt')
# torch.save(latent, 'tracing_test_data/network_from_latent_in.pt')
# torch.save(occ_hat, 'tracing_test_data/network_from_latent_out.pt')
# torch.save(occ_hat, 'tracing_test_data/predict_from_latent_out.pt')

In [14]:
# check test data
import torch

data_partial = torch.load('tracing_test_data/network_get_latent_in.pt', map_location=torch.device('cpu'))
# -> dict:
# {
#     'pts':        Tensor[1, 3, 10000]     (batch_size, 3d, gen_subsample_manifold),
#     'ids43':      Tensor[1, 156, 1]       (batch_size, gen_subsample_manifold // 64, 1), int64
#     'ids32':      Tensor[1, 625, 1]       (batch_size, gen_subsample_manifold // 16, 1), int64
#     'ids21':      Tensor[1, 2500, 1]      (batch_size, gen_subsample_manifold // 4, 1), int64
#     'ids10':      Tensor[1, 10000, 1]     (batch_size, gen_subsample_manifold, 1), int64
#     'support1':   Tensor[1, 3, 2500]      (batch_size, 3d, gen_subsample_manifold // 4)
#     'support2':   Tensor[1, 3, 625]       (batch_size, 3d, gen_subsample_manifold // 16)
#     'support3':   Tensor[1, 3, 156]       (batch_size, 3d, gen_subsample_manifold // 64)
#     'support4':   Tensor[1, 3, 39]        (batch_size, 3d, gen_subsample_manifold // 256)
#     'ids00':      Tensor[1, 10000, 16]    (batch_size, gen_subsample_manifold, InterpAttentionKHeadsNet.k), int64
#     'ids01':      Tensor[1, 2500, 16]     (batch_size, gen_subsample_manifold // 4, InterpAttentionKHeadsNet.k), int64
#     'ids11':      Tensor[1, 2500, 16]     (batch_size, gen_subsample_manifold // 4, InterpAttentionKHeadsNet.k), int64
#     'ids12':      Tensor[1, 625, 16]      (batch_size, gen_subsample_manifold // 16, InterpAttentionKHeadsNet.k), int64
#     'ids22':      Tensor[1, 625, 16]      (batch_size, gen_subsample_manifold // 16, InterpAttentionKHeadsNet.k), int64
#     'ids23':      Tensor[1, 156, 16]      (batch_size, gen_subsample_manifold // 64, InterpAttentionKHeadsNet.k), int64
#     'ids33':      Tensor[1, 156, 16]      (batch_size, gen_subsample_manifold // 64, InterpAttentionKHeadsNet.k), int64
#     'ids34':      Tensor[1, 39, 16]       (batch_size, gen_subsample_manifold // 256, InterpAttentionKHeadsNet.k), int64
#     'ids44':      Tensor[1, 39, 16]       (batch_size, gen_subsample_manifold // 256, InterpAttentionKHeadsNet.k), int64
#     'latents':    Tensor[1, 256, 10000]   (batch_size, network_latent_size, gen_subsample_manifold), float16
#     'proj_correction': None               (leftover from somewhere)
# }
partial_latent = torch.load('tracing_test_data/network_get_latent_out.pt', map_location=torch.device('cpu'))
# -> Tensor[1, 256, 10000] (batch_size, network_latent_size, gen_subsample_manifold)

latent = torch.load('tracing_test_data/network_from_latent_in.pt', map_location=torch.device('cpu'))
# -> dict:
# {
#     'pts_ms':             Tensor[1, 59979, 3]     (batch_size, num_points, 3d)
#     'normals_ms':         Tensor[1, 59979, 3]     (batch_size, num_points, 3d), float64
#     'pc_file_in':         str
#     'pts_query_ms':       Tensor[1, 0, 3]         (batch_size, num_query_points, 3d)
#     'imp_surf_dist_ms':   Tensor[1, 0, 3]         (batch_size, num_query_points, 3d)
#     'shape_id':           Tensor[1]               (batch_size, num_query_points, 3d), int64
#     'pts_raw_ms':         Tensor[1, 59979, 3]     (batch_size, num_points, 3d)
#     'pts':                Tensor[1, 3, 59979]     (batch_size, 3d, num_points)
#     'pts_query':          Tensor[1, 50000, 3]     (batch_size, num_query_points, 3d)
#     'occ':                Tensor[1, 0, 3]         (batch_size, num_query_points, 3d), int64
#     'ids43':              Tensor[1, 937, 1]       (batch_size, num_points // 64, 3d), int64
#     'ids32':              Tensor[1, 3748, 1]      (batch_size, num_points // 16, 3d), int64
#     'ids21':              Tensor[1, 14994, 1]     (batch_size, num_points // 4, 3d), int64
#     'ids10':              Tensor[1, 59979, 1]     (batch_size, num_points, 3d), int64
#     'support1':           Tensor[1, 3, 14994]     (batch_size, 3d, num_points // 4)
#     'support2':           Tensor[1, 3, 3748]      (batch_size, 3d, num_points // 16)
#     'support3':           Tensor[1, 3, 937]       (batch_size, 3d, num_points // 64)
#     'support4':           Tensor[1, 3, 234]       (batch_size, 3d, num_points // 256)
#     'ids00':              Tensor[1, 59979, 16]    (batch_size, num_query_points, InterpAttentionKHeadsNet.k), int64
#     'ids01':              Tensor[1, 14994, 16]    (batch_size, num_query_points // 4, InterpAttentionKHeadsNet.k), int64
#     'ids11':              Tensor[1, 14994, 16]    (batch_size, num_query_points // 4, InterpAttentionKHeadsNet.k), int64
#     'ids12':              Tensor[1, 3748, 16]     (batch_size, num_query_points // 16, InterpAttentionKHeadsNet.k), int64
#     'ids22':              Tensor[1, 3748, 16]     (batch_size, num_query_points // 16, InterpAttentionKHeadsNet.k), int64
#     'ids23':              Tensor[1, 937, 16]      (batch_size, num_query_points // 64, InterpAttentionKHeadsNet.k), int64
#     'ids33':              Tensor[1, 937, 16]      (batch_size, num_query_points // 64, InterpAttentionKHeadsNet.k), int64
#     'ids34':              Tensor[1, 234, 16]      (batch_size, num_query_points // 256, InterpAttentionKHeadsNet.k), int64
#     'ids44':              Tensor[1, 234, 16]      (batch_size, num_query_points // 256, InterpAttentionKHeadsNet.k), int64
#     'proj_ids':           Tensor[1, 50000, 64]    (batch_size, rec_batch_size, PocoModel.k), int64
#     'latents':            Tensor[1, 256, 59979]   (batch_size, network_latent_size, num_points)
#     'pts_local_ps':       Tensor[1, 50000, 50, 3] (batch_size, rec_batch_size, num_pts_local, 3d)
# }
# required for POCO inference: latents, proj_ids, pts, pts_query
# required for PPS inference:  pts_local_ps
# only train/val: pts_query_ms, imp_surf_dist_ms, occ
# POCO/PPS reconstruction and bookkeeping: 
# - pc_file_in: str, single point cloud file or dataset dir
# - shape_id: always 0 with batch_size 1
# - pts_raw_ms: point cloud without normalization
# Not used here: normals_ms (maybe in the future), ids and support (just there from the latent generation)
occ_hat = torch.load('tracing_test_data/network_from_latent_out.pt', map_location=torch.device('cpu'))
# -> Tensor[1, 2, 50000] (batch_size, in_out_probabilities, rec_batch_size)

# print key, type and shape if applicable for all items
print('data_partial:')
for key, value in data_partial.items():
    print_str = f'  {key}: {type(value)}'
    if isinstance(value, torch.Tensor):
        print_str += f', shape: {value.shape}, dtype: {value.dtype}'
        if value.numel() > 0:
            print_str += f', min: {value.min()}, max: {value.max()}'
    print(print_str)

print('latent:')
for key, value in latent.items():
    print_str = f'  {key}: {type(value)}'
    if isinstance(value, torch.Tensor):
        print_str += f', shape: {value.shape}, dtype: {value.dtype}'
        if value.numel() > 0:
            print_str += f', min: {value.min()}, max: {value.max()}'
    print(print_str)

data_partial:
  pts: <class 'torch.Tensor'>, shape: torch.Size([1, 3, 10000]), dtype: torch.float32, min: -0.4674682021141052, max: 0.4736515283584595
  ids43: <class 'torch.Tensor'>, shape: torch.Size([1, 156, 1]), dtype: torch.int64, min: 0, max: 38
  ids32: <class 'torch.Tensor'>, shape: torch.Size([1, 625, 1]), dtype: torch.int64, min: 0, max: 155
  ids21: <class 'torch.Tensor'>, shape: torch.Size([1, 2500, 1]), dtype: torch.int64, min: 0, max: 624
  ids10: <class 'torch.Tensor'>, shape: torch.Size([1, 10000, 1]), dtype: torch.int64, min: 0, max: 2499
  support1: <class 'torch.Tensor'>, shape: torch.Size([1, 3, 2500]), dtype: torch.float32, min: -0.4674682021141052, max: 0.4734063148498535
  support2: <class 'torch.Tensor'>, shape: torch.Size([1, 3, 625]), dtype: torch.float32, min: -0.45039722323417664, max: 0.4532862901687622
  support3: <class 'torch.Tensor'>, shape: torch.Size([1, 3, 156]), dtype: torch.float32, min: -0.4201790988445282, max: 0.43585532903671265
  support4: <cl

In [4]:
# # get hparams
# from source.ppsurf_model import PPSurfModel

# import yaml
# import inspect

# with open('models/ppsurf_50nn/version_0/config.yaml', 'r') as f:
#     config = yaml.safe_load(f)

# model_params = config.get('model', config)
# print(model_params)

# # get only the necessary ones
# sig = inspect.signature(PPSurfModel.__init__)
# valid_args = sig.parameters.keys()
# print(valid_args)
# filtered_config = {k: v for k, v in model_params.items() if k in valid_args}
# print(filtered_config)


In [5]:
# load model
from source.ppsurf_model import PPSurfModel
from source.ppsurf_data_loader import PPSurfDataModule
import pytorch_lightning as pl

model_kwargs = {
    'pointnet_latent_size': 256,  # Replace with your actual values
    'output_names': ['imp_surf_sign',],
    'in_channels': 3,
    'out_channels': 2,
    'k': 64,
    'lambda_l1': 0.0,
    'debug': False,
    'in_file': 'datasets/abc_minimal/04_pts_vis/00010009_d97409455fa543b3a224250f_trimesh_000.xyz.ply',
    'results_dir': './results_exporter',
    'padding_factor': 0.05,
    'name': 'ppsurf_50nn',
    'network_latent_size': 256,
    'gen_subsample_manifold_iter': 10,
    'gen_subsample_manifold': 10000,
    'gen_resolution_global': 257,
    'num_pts_local': 50,
    'rec_batch_size': 50000,
    'gen_refine_iter': 10,
    'workers': 8
}

model = PPSurfModel.load_from_checkpoint(
    'models/ppsurf_50nn/version_0/checkpoints/last.ckpt',
    **model_kwargs)
model.eval()

print(model)

InterpNet - Simple - K=64
Network -- backbone -- 12798516 parameters
Network -- projection -- 280898 parameters
InterpNet - Simple - K=64
Network -- backbone -- 12798516 parameters
Network -- projection -- 346176 parameters
Network -- point_net -- 471297 parameters
Network -- mlp -- 133122 parameters
PPSurfModel(
  (network): PPSurfNetwork(
    (encoder): FKAConvNetwork(
      (cv0): FKAConvLayer(
        (cv): Conv2d(3, 64, kernel_size=(1, 16), stride=(1, 1), bias=False)
        (fc1): Conv2d(3, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (fc2): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (fc3): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): InstanceNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
        (bn2): InstanceNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
        (activation): SiLU()
      )
      (bn0): BatchNorm1d(64, eps=1e-05, momentum=0.1, affi

In [13]:
import torch
import torch_cluster
from torch.export import export, draft_export
from executorch.exir import to_edge_transform_and_lower

# 1. Grab the internal network from your loaded Lightning model
network = model.network.cpu().eval()
device = network.device
print(f"Network device: {device}")

test = torch.randn(1, 3, 10000).to(network.device)

# 2. Define Wrappers for the two specific methods
class GetLatentWrapper(torch.nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net
    def forward(self, data):
        # spectral_only=True so no knn ids are computed, since that would not be supported for tracing
        
        latents = self.net.encoder.forward(data, spectral_only=True)
        return latents

class FromLatentWrapper(torch.nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net
    def forward(self, latent_tensor):
        # We wrap the tensor back into the dictionary the method expects
        return self.net.from_latent({"latent": latent_tensor})

# 3. Define Example Inputs (Match your model's expected shapes)
num_pts_raw = 59979  # from the test data
num_pts_subsample = 10000  # gen_subsample_manifold
example_input_get_latent = {
    'pts': torch.randn(1, 3, num_pts_subsample, device=device),
    'ids43': torch.randint(0, 39, (1, 937, 1), device=device),
    'ids32': torch.randint(0, 156, (1, 3748, 1), device=device),
    'ids21': torch.randint(0, 625, (1, 14994, 1), device=device),
    'ids10': torch.randint(0, 2500, (1, num_pts_subsample, 1), device=device),
    'support1': torch.randn(1, 3, 14994, device=device),
    'support2': torch.randn(1, 3, 3748, device=device),
    'support3': torch.randn(1, 3, 937, device=device),
    'support4': torch.randn(1, 3, 234, device=device),
    'ids00': torch.randint(0, num_pts_subsample, (1, num_pts_subsample, 16), device=device),
    'ids01': torch.randint(0, num_pts_subsample, (1, 14994, 16), device=device),
    'ids11': torch.randint(0, 2500, (1, 14994, 16), device=device),
    'ids12': torch.randint(0, 2500, (1, 3748, 16), device=device),
    'ids22': torch.randint(0, 625, (1, 3748, 16), device=device),
    'ids23': torch.randint(0, 625, (1, 937, 16), device=device),
    'ids33': torch.randint(0, 156, (1, 937, 16), device=device),
    'ids34': torch.randint(0, 156, (1, 234, 16), device=device),
    'ids44': torch.randint(0, 39, (1, 234, 16), device=device),
    'pts_query': torch.randn(1, 10, 3, device=device),
    'proj_ids': torch.randint(0, num_pts_raw, (1, 50000, 64), device=device),
}
example_input_from_latent = {
    'latent': torch.randn(1, 256, num_pts_raw, device=device),
    'proj_ids': torch.randint(0, num_pts_raw, (1, 50000, 64), device=device),
    'pts': torch.randn(1, 3, num_pts_raw, device=device),
    'pts_query': torch.randn(1, 10, 3, device=device),
}

# 4. Capture and Lower
# We use the new 2026 standard flow: Export -> To Edge/Lower -> To Executorch
def save_pte(wrapper_instance, example_args, filename):
    # Step A: Capture into ATen Dialect
    # exported_program = export(wrapper_instance, (example_args,))
    exported_program = draft_export(wrapper_instance, (example_args,))  # for debug info
    
    # Step B: Lower to Edge Dialect and convert to PTE
    # This is the modern replacement for 'capture_program' or 'to_edge'
    pte = to_edge_transform_and_lower(exported_program).to_executorch()
    
    with open(filename, "wb") as f:
        f.write(pte.buffer)
    print(f"Successfully saved {filename}")

# Execute the saves
save_pte(GetLatentWrapper(network), example_input_get_latent, "get_latent.pte")
save_pte(FromLatentWrapper(network), example_input_from_latent, "from_latent.pte")

Network device: cpu


RuntimeError: Tried to cast object to type __torch__.torch.classes.profiler._RecordFunction (of Python compilation unit at: 0000000000000000) but object was missing attribute capsule

In [ ]:
# load and test exported model
